In [ ]:
# 1. Import the pandas library
import pandas as pd

# 2. Load your uploaded file into a pandas DataFrame
# --- CHANGE FILENAME HERE IF NEEDED ---
file_path = '/content/MASTERCLASS 1 - Clean Data.csv'
df = pd.read_csv(file_path)

# 3. Display the first 5 rows
print("--- FIRST 5 ROWS ---")
display(df.head(5))

# 4. Display all column names
print("\n--- COLUMN NAMES ---")
print(df.columns.tolist())

# 5. Display the number of rows and columns
print("\n--- NUMBER OF ROWS AND COLUMNS ---")
rows, columns = df.shape
print(f"Rows: {rows}")
print(f"Columns: {columns}")

# 6. Display the data types of all columns
print("\n--- DATA TYPES OF ALL COLUMNS ---")
print(df.dtypes)

--- FIRST 5 ROWS ---


,Order_ID,Order_Date,Customer_ID,Unnamed: 3,Product,Category,Region,Quantity,Revenue,Profit
0,1001,01/06/2026,C001,New Customer,Laptop,Electronics,Delhi,1,65000,8000
1,1002,03/06/2026,C002,Repeat Customer,Phone,Electronics,Delhi,2,50000,6000
2,1003,03/03/2026,C003,Repeat Customer,Headphones,Electronics,Mumbai,1,5000,1200
3,1004,05/06/2026,C004,Repeat Customer,Chair,Furniture,Bangalore,3,12000,2500
4,1005,03/07/2026,C314,Repeat Customer,Vacuum Cleaner,Home,Delhi,0,8506,1455



--- COLUMN NAMES ---
['Order_ID', 'Order_Date', 'Customer_ID', 'Unnamed: 3', 'Product', 'Category', 'Region', 'Quantity', 'Revenue', 'Profit']

--- NUMBER OF ROWS AND COLUMNS ---
Rows: 2000
Columns: 10

--- DATA TYPES OF ALL COLUMNS ---
Order_ID        int64
Order_Date     object
Customer_ID    object
Unnamed: 3     object
Product        object
Category       object
Region         object
Quantity        int64
Revenue        object
Profit          int64
dtype: object


In [ ]:
# Check for missing (null) values in each column
print("--- MISSING VALUES PER COLUMN ---")
missing_counts = df.isnull().sum()
print(missing_counts)

--- MISSING VALUES PER COLUMN ---
Order_ID       0
Order_Date     0
Customer_ID    0
Unnamed: 3     0
Product        7
Category       0
Region         0
Quantity       0
Revenue        0
Profit         0
dtype: int64


In [ ]:
# 1. Prepare and clean data types
# Ensure the Order_Date is converted to a proper datetime format
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce', dayfirst=True)

# Ensure the Revenue column is in numeric format
if df['Revenue'].dtype == 'object':
    df['Revenue'] = df['Revenue'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
    df['Revenue'] = pd.to_numeric(df['Revenue'], errors='coerce')

# 2. Group by Customer_ID and calculate the requested metrics
customer_df = df.groupby('Customer_ID').agg(
    Order_Count=('Order_ID', 'nunique'),
    Total_Revenue=('Revenue', 'sum'),
    Avg_Order_Value=('Revenue', 'mean'),
    First_Purchase=('Order_Date', 'min'),
    Last_Purchase=('Order_Date', 'max')
).reset_index()

# 3. Display the first 10 rows
print("--- FIRST 10 ROWS OF CUSTOMER DATASET ---")
display(customer_df.head(10))

# 4. Display the total number of customers
print("\n--- TOTAL NUMBER OF UNIQUE CUSTOMERS ---")
print(len(customer_df))

# 5. Display the final column names
print("\n--- COLUMN NAMES ---")
print(customer_df.columns.tolist())

--- FIRST 10 ROWS OF CUSTOMER DATASET ---


,Customer_ID,Order_Count,Total_Revenue,Avg_Order_Value,First_Purchase,Last_Purchase
0,C000,26,1900945,73113.269231,2026-01-02,2026-08-13
1,C001,1,65000,65000.000000,2026-06-01,2026-06-01
2,C002,9,256033,28448.111111,2026-02-17,2026-08-06
3,C003,3,82737,27579.000000,2026-03-02,2026-06-06
4,C004,5,79072,15814.400000,2026-01-03,2026-07-06
5,C005,2,49630,24815.000000,2026-07-04,2026-08-12
6,C006,3,228080,76026.666667,2026-01-11,2026-08-19
7,C007,3,99452,33150.666667,2026-04-19,2026-07-29
8,C008,1,76659,76659.000000,2026-04-01,2026-04-01
9,C009,3,143415,47805.000000,2026-04-25,2026-06-07



--- TOTAL NUMBER OF UNIQUE CUSTOMERS ---
416

--- COLUMN NAMES ---
['Customer_ID', 'Order_Count', 'Total_Revenue', 'Avg_Order_Value', 'First_Purchase', 'Last_Purchase']


In [ ]:
# 1. Determine the reference date (the maximum order date in the original transaction dataset)
reference_date = df['Order_Date'].max()
print(f"Reference Date (Latest Order Date in Dataset): {reference_date.strftime('%Y-%m-%d')}")

# 2. Calculate RFM features on the customer dataset
# Recency: Difference in days between reference date and the customer's last purchase
customer_df['Recency'] = (reference_date - customer_df['Last_Purchase']).dt.days

# Frequency: Map directly from our calculated Order_Count
customer_df['Frequency'] = customer_df['Order_Count']

# Monetary: Map directly from our calculated Total_Revenue
customer_df['Monetary'] = customer_df['Total_Revenue']

# 3. Display the first 10 rows of the updated dataset
print("\n--- CUSTOMER-LEVEL DATASET WITH RFM FEATURES ---")
display(customer_df.head(10))

Reference Date (Latest Order Date in Dataset): 2026-12-01

--- CUSTOMER-LEVEL DATASET WITH RFM FEATURES ---


,Customer_ID,Order_Count,Total_Revenue,Avg_Order_Value,First_Purchase,Last_Purchase,Recency,Frequency,Monetary
0,C000,26,1900945,73113.269231,2026-01-02,2026-08-13,110,26,1900945
1,C001,1,65000,65000.000000,2026-06-01,2026-06-01,183,1,65000
2,C002,9,256033,28448.111111,2026-02-17,2026-08-06,117,9,256033
3,C003,3,82737,27579.000000,2026-03-02,2026-06-06,178,3,82737
4,C004,5,79072,15814.400000,2026-01-03,2026-07-06,148,5,79072
5,C005,2,49630,24815.000000,2026-07-04,2026-08-12,111,2,49630
6,C006,3,228080,76026.666667,2026-01-11,2026-08-19,104,3,228080
7,C007,3,99452,33150.666667,2026-04-19,2026-07-29,125,3,99452
8,C008,1,76659,76659.000000,2026-04-01,2026-04-01,244,1,76659
9,C009,3,143415,47805.000000,2026-04-25,2026-06-07,177,3,143415


### Define Churn Status and Count Customers
We will define a customer as **Churned** (Value: `1` or `'Churned'`) if they meet all three conditions simultaneously:
1. **Recency** is 180 days or more (`>= 180`)
2. **Frequency** is less than 5 (`< 5`)
3. **Monetary** is less than 100,000 (`< 100000`)

Otherwise, they are marked as **Active** or **Not Churned** (`0` or `'Not Churned'`).

In [ ]:
# 1. Apply the user's explicit multi-condition rules to label churn status
# We check: Recency >= 180 AND Frequency < 5 AND Monetary < 100,000
churn_condition = (
    (customer_df['Recency'] >= 180) &
    (customer_df['Frequency'] < 5) &
    (customer_df['Monetary'] < 100000)
)

# Label 'Churned' if condition is met, otherwise 'Not Churned'
customer_df['Churn_Status'] = 'Not Churned'
customer_df.loc[churn_condition, 'Churn_Status'] = 'Churned'

# 2. Count how many customers fall into each category
churn_counts = customer_df['Churn_Status'].value_counts()
print("--- CHURN STATUS COUNTS ---")
print(churn_counts)

# 3. Preview the updated customer dataset to confirm changes
print("\n--- UPDATED CUSTOMER DATASET PREVIEW (FIRST 10 ROWS) ---")
display(customer_df[['Customer_ID', 'Recency', 'Frequency', 'Monetary', 'Churn_Status']].head(10))

--- CHURN STATUS COUNTS ---
Churn_Status
Not Churned    317
Churned         99
Name: count, dtype: int64

--- UPDATED CUSTOMER DATASET PREVIEW (FIRST 10 ROWS) ---


,Customer_ID,Recency,Frequency,Monetary,Churn_Status
0,C000,110,26,1900945,Not Churned
1,C001,183,1,65000,Churned
2,C002,117,9,256033,Not Churned
3,C003,178,3,82737,Not Churned
4,C004,148,5,79072,Not Churned
5,C005,111,2,49630,Not Churned
6,C006,104,3,228080,Not Churned
7,C007,125,3,99452,Not Churned
8,C008,244,1,76659,Churned
9,C009,177,3,143415,Not Churned


### Step 1: Inspect the Order Date Range
We will check the earliest and latest order dates available in our dataset to construct a scientifically sound, leakage-free cutoff.

In [ ]:
min_date = df['Order_Date'].min()
max_date = df['Order_Date'].max()
print(f"Earliest date in dataset: {min_date.strftime('%Y-%m-%d')}")
print(f"Latest date in dataset:   {max_date.strftime('%Y-%m-%d')}")
print(f"Total span of dataset:    {(max_date - min_date).days} days")

Earliest date in dataset: 2026-01-01
Latest date in dataset:   2026-12-01
Total span of dataset:    334 days


### Step 2: Establish a Split and Calculate Leakage-Safe Features and Target

Given the dataset spans exactly 1 year (2026-01-01 to 2026-12-01, ~334 days):
* We will set our **Cutoff Date** at **2026-06-01** (approximately the halfway mark).
* **Historical Period:** Orders from `2026-01-01` to `2026-05-31` (used strictly for feature calculations).
* **Future Observation Period:** Orders from `2026-06-01` to `2026-12-01` (used strictly for identifying if a customer makes a purchase). Since this future window is 184 days long, any customer who makes **zero** purchases in this window has went 180+ days without a purchase, classifying them as **Churned**.

In [ ]:
# 1. Split the transactions using our cutoff date
cutoff_date = pd.to_datetime('2026-06-01')

historical_df = df[df['Order_Date'] < cutoff_date].copy()
future_df = df[df['Order_Date'] >= cutoff_date].copy()

# 2. Build Historical Features (using only historical data)
# Recency reference date is the last day of the historical period (2026-05-31)
hist_ref_date = pd.to_datetime('2026-05-31')

features_df = historical_df.groupby('Customer_ID').agg(
    Historical_Last_Purchase=('Order_Date', 'max'),
    Frequency=('Order_ID', 'nunique'),
    Monetary=('Revenue', 'sum'),
    Avg_Order_Value=('Revenue', 'mean')
).reset_index()

# Recency = Days since last historical purchase relative to historical end date
features_df['Recency'] = (hist_ref_date - features_df['Historical_Last_Purchase']).dt.days

# 3. Determine Churn Status in the Future Period
# Customers who purchased in the future period
active_future_customers = set(future_df['Customer_ID'].unique())

# If a customer from history has no purchases in the future_df, they are labeled Churned (1), else Active (0)
features_df['Churn_Status'] = features_df['Customer_ID'].apply(
    lambda x: 'Churned' if x not in active_future_customers else 'Not Churned'
)

# Clean up intermediate columns to keep the dataset simple and clean
final_ml_df = features_df[['Customer_ID', 'Recency', 'Frequency', 'Monetary', 'Avg_Order_Value', 'Churn_Status']].copy()

# 4. Show Churn Balance Counts
print("--- LEAKAGE-SAFE CHURN COUNTS ---")
print(final_ml_df['Churn_Status'].value_counts())

# 5. Display the final prepared dataset
print("\n--- FINAL PREPARED CUSTOMER-LEVEL DATASET (FIRST 10 ROWS) ---")
display(final_ml_df.head(10))

--- LEAKAGE-SAFE CHURN COUNTS ---
Churn_Status
Not Churned    243
Churned        135
Name: count, dtype: int64

--- FINAL PREPARED CUSTOMER-LEVEL DATASET (FIRST 10 ROWS) ---


,Customer_ID,Recency,Frequency,Monetary,Avg_Order_Value,Churn_Status
0,C000,1,14,683127,48794.785714,Not Churned
1,C002,3,2,28269,14134.500000,Not Churned
2,C003,89,2,73499,36749.500000,Not Churned
3,C004,31,3,65411,21803.666667,Not Churned
4,C006,140,1,19130,19130.000000,Not Churned
5,C007,42,1,17340,17340.000000,Not Churned
6,C008,60,1,76659,76659.000000,Churned
7,C009,22,2,124103,62051.500000,Not Churned
8,C010,90,3,32564,10854.666667,Not Churned
9,C012,8,14,509601,36400.071429,Not Churned
